# Elasticsearch for Pin Full-Text Search

Research goal: set up Elasticsearch 8.x index for pins, tune BM25 analysis, and measure recall.


In [ ]:
# Requires Elasticsearch running locally or via Docker:
# docker run -d -p 9200:9200 -e "discovery.type=single-node" -e "xpack.security.enabled=false" \
#   docker.elastic.co/elasticsearch/elasticsearch:8.13.0
# !pip install elasticsearch==8.13.0


In [ ]:
from elasticsearch import Elasticsearch
import json, random, time
import pandas as pd
import matplotlib.pyplot as plt

es = Elasticsearch("http://localhost:9200")
print("ES cluster info:", es.info()["version"]["number"])


## 1. Index setup with custom Russian analyzer

In [ ]:
INDEX = "pins_research"

if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)

es.indices.create(index=INDEX, body={
    "settings": {
        "analysis": {
            "analyzer": {
                "ru_en_analyzer": {
                    "type": "custom",
                    "tokenizer": "standard",
                    "filter": ["lowercase", "russian_morphology", "english_morphology", "russian_stop"]
                }
            }
        }
    },
    "mappings": {
        "properties": {
            "pin_id":      {"type": "keyword"},
            "title":       {"type": "text", "analyzer": "ru_en_analyzer"},
            "description": {"type": "text", "analyzer": "ru_en_analyzer"},
            "location":    {"type": "text"},
            "tags":        {"type": "keyword"},
            "created_at":  {"type": "date"}
        }
    }
})
print(f"Index {INDEX!r} created.")


## 2. Index synthetic pin documents

In [ ]:
pins = [
    {"pin_id": "p001", "title": "Закат на Байкале", "description": "Невероятные оттенки оранжевого на воде", "location": "Байкал, Россия", "tags": ["природа", "закат", "вода"]},
    {"pin_id": "p002", "title": "Уличная еда в Стамбуле", "description": "Симит, балык экмек и турецкий чай на набережной", "location": "Стамбул, Турция", "tags": ["еда", "улица", "турция"]},
    {"pin_id": "p003", "title": "Горы Алтая", "description": "Катунь в сентябре — бирюзовая вода и горные вершины", "location": "Алтай, Россия", "tags": ["горы", "поход", "россия"]},
    {"pin_id": "p004", "title": "Ночная Москва", "description": "Арт-объекты и неоновые надписи в Артплее", "location": "Москва, Россия", "tags": ["ночь", "арт", "москва"]},
    {"pin_id": "p005", "title": "Santorini Sunset", "description": "White-washed cliffs and caldera view at golden hour", "location": "Santorini, Greece", "tags": ["sunset", "travel", "greece"]},
    {"pin_id": "p006", "title": "Tokyo Night Photography", "description": "Neon lights and rain reflections in Shinjuku", "location": "Tokyo, Japan", "tags": ["night", "japan", "photography"]},
    {"pin_id": "p007", "title": "Dolomites Hiking", "description": "Tre Cime loop trail on an August morning", "location": "Dolomites, Italy", "tags": ["hiking", "mountains", "italy"]},
    {"pin_id": "p008", "title": "Melbourne Coffee", "description": "Specialty coffee in hidden laneway cafes", "location": "Melbourne, Australia", "tags": ["coffee", "cafe", "australia"]},
]

for pin in pins:
    es.index(index=INDEX, id=pin["pin_id"], document=pin)
es.indices.refresh(index=INDEX)
print(f"Indexed {len(pins)} pins.")


## 3. BM25 search experiments

In [ ]:
def search_pins(query, size=5, fields=("title^2", "description", "location", "tags")):
    resp = es.search(index=INDEX, body={
        "size": size,
        "query": {"multi_match": {"query": query, "fields": list(fields)}}
    })
    return [(h["_id"], round(h["_score"], 3), h["_source"]["title"]) for h in resp["hits"]["hits"]]

test_queries = [
    ("закат вода",     "p001"),
    ("горный поход",   "p003"),
    ("ночное фото",    "p004"),
    ("sunset travel",  "p005"),
    ("coffee cafe",    "p008"),
    ("hiking trail",   "p007"),
]

hits_at_1 = 0
for q, expected in test_queries:
    hits = search_pins(q)
    if hits and hits[0][0] == expected:
        hits_at_1 += 1
    print(f"Q: {q!r:25s} => top: {hits[0] if hits else None} [expected={expected}]")

print(f"
Hits@1: {hits_at_1}/{len(test_queries)}")


## 4. Explain BM25 scoring

In [ ]:
resp = es.explain(index=INDEX, id="p001", body={
    "query": {"multi_match": {"query": "закат вода", "fields": ["title^2", "description"]}}
})
print(json.dumps(resp["explanation"], indent=2, ensure_ascii=False))


## Conclusions

- BM25 works well for exact-keyword and morphology-aware queries
- Russian analyzer (russian_morphology filter) is essential for stemming
- title^2 boost gives correct prioritization
- **Gap**: BM25 fails for semantic / paraphrase queries ("снимок природы" ≠ "закат на Байкале") — handled by Qdrant vector search in hybrid pipeline
